In [50]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage, SystemMessage

In [51]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0.1
)

llm1 = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 1.2
)
llm2 = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0.7
)

In [52]:
class EvalutionSchema(BaseModel):
    status : Literal["approved", "not_approved"] = Field(..., description="provide the status of approval of speech")
    feedback : str = Field(..., description="feedback of the generated speech message")

In [ ]:
class WorkflowInput(TypedDict):
    topic : str
    status : Literal["approved", "not_approved"]
    feedback : str
    generated_message : str
    iteration : int
    max_iteration : int


In [54]:
evalution_model = llm.with_structured_output(EvalutionSchema)
generation_model = llm1
optimiser_model = llm2

In [55]:
def speech_func(state : WorkflowInput):
    prompt = f""" you are a high professional expect for speech generation script for international submit, take the topic name form {state["topic"]} and generate the high qulaity speech """
    response = generation_model.invoke(prompt).content
    return {
        "generated_message" : response
        
    }

In [56]:
def evalution_func(state : WorkflowInput):
    prompt = f""" you are a high quality international speech professor expect, read the generated speech message {state["generated_message"]} from the topic {state["topic"]} and Evaluate the the generated speech and give the response is speech approved or not, also provide the feedback of speech : \n
    rules to follow must and should : 
    1.speech should be under 300 charcters
    2.Its should high quality internation standards language grammar
    3.doesnt effect any region or any other thing
    4.high quality story telling with professional expertise"""
    response = evalution_model.invoke(prompt)
    return{
        "status" : response.status,
        "feedback" : response.feedback
    }

In [57]:
def optimizer_func(state : WorkflowInput):
    prompt = f""" follow the generated speech {state["generated_message"]} and understand the speech and take the feedback from {state["feedback"]} and rewrite the speech"""
    iteration_count = state["iteration"] + 1
    response = optimiser_model.invoke(prompt)
    return{
        "generated_message" : response.content,
        "iteration" : iteration_count
    }

In [58]:
def approval_check(state : WorkflowInput) -> Literal["approved", "not_approved"]:
    status = state["status"]
    if status == "approved" or state["max_iteration"] == state["iteration"]:
        return "approved"
    else:
        return "not_approved"

In [59]:
graph = StateGraph(WorkflowInput)

graph.add_node("speech_generation", speech_func)
graph.add_node("evalution_work", evalution_func)
graph.add_node("optimizer_generation", optimizer_func)

graph.add_edge(START, "speech_generation")
graph.add_edge("speech_generation", "evalution_work")
graph.add_conditional_edges("evalution_work", approval_check, {"approved" : END, "not_approved" : "optimizer_generation"})
graph.add_edge("optimizer_generation", "evalution_work")

workflow = graph.compile()


In [66]:
initial_state = {
    "topic" : "artifical intelligence",
    "iteration" : 0,
    "max_iteration" : 3
}
finial_state =  workflow.invoke(initial_state)
print(finial_state)


{'topic': 'artifical intelligence', 'status': 'approved', 'feedback': 'The speech meets all the stipulated criteria: it stays well under the 300‑character limit, uses polished international English, avoids regional bias, and presents a concise, compelling narrative with clear professional authority. Minor tightening (e.g., removing a filler word) could make it even sharper, but it is already high‑quality and suitable for a global audience.', 'generated_message': '**AI – New Frontier**  \n\nGood morning. AI is the new catalyst for progress, boosting GDP, saving lives, and protecting climate. Yet it risks bias, loss of privacy, and inequality. We must set global standards, embed ethics, and educate all. Let us build AI that serves humanity, not oppresses it. Thank you.', 'iteration': 1, 'max_iteration': 3}
